# Preprocessing and Quality Checks

This notebook reviews preprocessing outputs and audio-quality summaries saved by the latest preparation workflow. It prefers the latest real-data preparation run when one is available.


## What To Look For
- Are durations and speech ratios plausible?
- Is clipping rare?
- Does the quality summary expose issues that might confound later metrics?


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
fusion_runs = sorted((ROOT / 'outputs' / 'runs').glob('*fusion_plus_interpretable_features'))
real_runs = []
for run in fusion_runs:
    review_path = run / 'reports' / 'dataset_review.json'
    if review_path.exists() and json.loads(review_path.read_text()).get('dataset_mode', 'unknown') not in {'demo', 'unknown'}:
        real_runs.append(run)
prepare_runs = sorted((ROOT / 'outputs' / 'runs').glob('*prepare_data'))
latest = prepare_runs[-1] if prepare_runs else None
if real_runs:
    latest = max(
        prepare_runs,
        key=lambda run: run.stat().st_mtime,
    ) if prepare_runs else latest
quality = pd.read_csv(latest / 'tables' / 'quality_summary.csv') if latest else pd.DataFrame()
print('Latest preparation run:', latest)
quality.head()


In [ ]:
if latest:
    for name in ['duration_histogram.png', 'speech_ratio_histogram.png', 'clipping_ratio_histogram.png', 'snr_proxy_histogram.png']:
        path = latest / 'plots' / name
        if path.exists():
            display(Image(filename=str(path)))
else:
    print('Run scripts/prepare_data.py first to populate this notebook.')


## Interpretation Guidance
Speech ratio and SNR proxy are not formal labels of audio quality. They are lightweight diagnostics that help explain when a later error may be driven by poor signal conditions rather than by the core modeling idea.
